# Bài tập - Giải thuật phát hiện cạnh trong xử lý ảnh
**BVU - Cao học - Xử lý ảnh - Trương Đình Phúc**

So sánh 5 toán tử phát hiện cạnh: **Roberts, Prewitt, Sobel, Laplacian, Canny**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0. Chuẩn bị dữ liệu

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

image_path = '/content/drive/MyDrive/Colab Notebooks/house.tiff'
image = cv2.imread(image_path)

if image is None:
    raise FileNotFoundError("Khong tim thay anh hoac sai duong dan.")

print("Kich thuoc anh:", image.shape)

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); axs[0].set_title("Anh goc"); axs[0].axis("off")
axs[1].imshow(gray, cmap="gray"); axs[1].set_title("Anh xam"); axs[1].axis("off")
axs[2].imshow(blurred, cmap="gray"); axs[2].set_title("Gaussian Blur"); axs[2].axis("off")
plt.tight_layout()
plt.show()

## 1. Roberts Operator

In [ ]:
def roberts_edge(img):
    kernel_x = np.array([[1, 0], [0, -1]], dtype=np.float32)
    kernel_y = np.array([[0, 1], [-1, 0]], dtype=np.float32)

    gx = cv2.filter2D(img, cv2.CV_32F, kernel_x)
    gy = cv2.filter2D(img, cv2.CV_32F, kernel_y)

    magnitude = cv2.magnitude(gx, gy)
    magnitude = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)
    return magnitude.astype(np.uint8)

roberts = roberts_edge(gray)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(gray, cmap='gray'); plt.title("Anh xam"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(roberts, cmap='gray'); plt.title("Roberts Edge"); plt.axis("off")
plt.tight_layout()
plt.show()

## 2. Prewitt Operator

In [ ]:
def prewitt_edge(img):
    kernel_x = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32)
    kernel_y = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=np.float32)

    gx = cv2.filter2D(img, cv2.CV_32F, kernel_x)
    gy = cv2.filter2D(img, cv2.CV_32F, kernel_y)

    magnitude = cv2.magnitude(gx, gy)
    magnitude = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)
    return magnitude.astype(np.uint8)

prewitt = prewitt_edge(gray)

plt.figure(figsize=(6, 6))
plt.imshow(prewitt, cmap='gray')
plt.title("Prewitt Edge Detection")
plt.axis("off")
plt.show()

## 3. Sobel Operator

In [ ]:
gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)

sobelx = cv2.convertScaleAbs(gx)
sobely = cv2.convertScaleAbs(gy)

sobel_combined = cv2.magnitude(gx, gy)
sobel_combined = cv2.normalize(sobel_combined, None, 0, 255, cv2.NORM_MINMAX)
sobel_combined = sobel_combined.astype(np.uint8)

fig, axs = plt.subplots(1, 4, figsize=(20, 5))
axs[0].imshow(gray, cmap='gray'); axs[0].set_title("Anh xam"); axs[0].axis("off")
axs[1].imshow(sobelx, cmap='gray'); axs[1].set_title("Sobel X"); axs[1].axis("off")
axs[2].imshow(sobely, cmap='gray'); axs[2].set_title("Sobel Y"); axs[2].axis("off")
axs[3].imshow(sobel_combined, cmap='gray'); axs[3].set_title("Sobel Magnitude"); axs[3].axis("off")
plt.tight_layout()
plt.show()

## 4. Laplacian Operator

In [ ]:
laplacian = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
laplacian = cv2.convertScaleAbs(laplacian)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(gray, cmap='gray'); plt.title("Anh xam"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(laplacian, cmap='gray'); plt.title("Laplacian"); plt.axis("off")
plt.tight_layout()
plt.show()

## 5. Canny Edge Detector

In [ ]:
canny_low = cv2.Canny(gray, 50, 150)
canny_high = cv2.Canny(gray, 100, 200)
canny_optimal = cv2.Canny(blurred, 50, 150, L2gradient=True)

fig, axs = plt.subplots(1, 4, figsize=(20, 5))
axs[0].imshow(gray, cmap='gray'); axs[0].set_title("Anh xam"); axs[0].axis("off")
axs[1].imshow(canny_low, cmap='gray'); axs[1].set_title("Canny (50,150)"); axs[1].axis("off")
axs[2].imshow(canny_high, cmap='gray'); axs[2].set_title("Canny (100,200)"); axs[2].axis("off")
axs[3].imshow(canny_optimal, cmap='gray'); axs[3].set_title("Canny + Gaussian Blur"); axs[3].axis("off")
plt.tight_layout()
plt.show()

## So sánh & nhận xét

| Toán tử | Kích thước kernel | Đặc điểm |
|---|---|---|
| Roberts | 2x2 | Nhanh, nhạy nhiễu, bắt cạnh chéo tốt |
| Prewitt | 3x3 | Đơn giản, làm mượt nhẹ theo hướng vuông góc với gradient |
| Sobel | 3x3 | Có trọng số ở tâm (1-2-1), chống nhiễu tốt hơn Prewitt |
| Laplacian | 3x3 (đạo hàm bậc 2) | Nhạy nhiễu cao, thường cần làm mượt Gaussian trước |
| Canny | Nhiều bước (Gaussian + gradient + non-max suppression + hysteresis) | Cho cạnh mảnh, liền mạch, ít nhiễu nhất trong 5 toán tử |

**Kết luận:** Roberts/Prewitt/Sobel chỉ là đạo hàm bậc 1 nên nhạy nhiễu và cho cạnh dày; Laplacian là đạo hàm bậc 2 nên càng nhạy nhiễu hơn nếu không làm mượt trước; Canny kết hợp làm mượt, tính gradient, làm mảnh cạnh (non-max suppression) và ngưỡng kép (hysteresis) nên cho kết quả cạnh rõ và sạch nhất, phù hợp cho các ứng dụng thực tế như phát hiện biển số, phát hiện vật thể.